# 01 - Eksplorasi Dataset UCI HAR

Notebook ini hanya **membaca** artefak yang sudah dihasilkan oleh pipeline
(`python scripts/prepare_data.py --download`). Tidak ada model yang dilatih di sini.

Sumber data: `reports/tables/dataset_summary.json` dan
`reports/tables/dataset_verification.json`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

summary = json.loads((TABLES / "dataset_summary.json").read_text(encoding="utf-8"))
verification = json.loads((TABLES / "dataset_verification.json").read_text(encoding="utf-8"))
print("sumber:", summary["source_label"])
print("sampel:", summary["n_samples"], "| fitur:", summary["n_features"], "| subjek:", summary["n_subjects"])

## Distribusi kelas

In [ ]:
class_table = pd.DataFrame(summary["class_distribution"])
display(class_table)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_table["activity"], class_table["n"], color="#2a6f97")
ax.set_ylabel("Jumlah sampel")
ax.set_title("Distribusi kelas aktivitas")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURES / "notebook_01_class_distribution.png", dpi=150)
plt.show()

## Sampel per subjek

Tanda `*` menandai subjek yang berada pada partisi uji resmi.

In [ ]:
subjects = pd.DataFrame(summary["subjects"])
subjects["split"] = subjects["split"].map({"train": "latih", "test": "uji"})
display(subjects.head(10))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(subjects["subject"].astype(str), subjects["n_samples"], color="#e07a5f")
ax.set_xlabel("Subjek")
ax.set_ylabel("Jumlah sampel")
ax.set_title("Jumlah sampel per subjek")
fig.tight_layout()
fig.savefig(FIGURES / "notebook_01_samples_per_subject.png", dpi=150)
plt.show()

## Komposisi fitur

In [ ]:
groups = summary.get("feature_groups", {})
sensor_counts = groups.get("sensor_counts", {})
print("jumlah fitur:", summary["n_features"])
print("domain:", groups.get("domain_counts"))
print("sensor:", sensor_counts)
print("catatan:", groups.get("note"))

## Verifikasi integritas

In [ ]:
checks = verification["checks"]
for key in ("n_features", "n_train", "n_test", "n_subjects", "subject_overlap", "duplicate_rows", "missing_values", "n_duplicate_feature_names"):
    print(f"{key:28s}: {checks.get(key)}")
print("peringatan:", verification.get("warnings"))

**Catatan:** 561 fitur adalah fitur prakomputasi oleh pembuat dataset. Pipeline ini
tidak mengekstrak ulang fitur tersebut; sinyal mentah 128x9 hanya dipakai model 1D-CNN.
Nama fitur duplikat pada `features.txt` diberi sufiks `__dupN` tanpa mengubah urutan kolom.